# JSONBench walkthrough

Generate a small travel dataset in memory.

In [ ]:
from json_bench.data import generate_travel_dataset

dataset = generate_travel_dataset(
    scale=0.01,
    seed=23,
    reference_date="2025-11-01",
)

## Deploy the databases

Start Docker and run `uv sync --locked` first. BONDI waits for each database to be ready.
Couchbase uses Community Edition with a `travel` bucket. Change the names or ports if needed.

In [ ]:
from bondi.deployment import deploy_couchbase, deploy_mongodb

mongo_deployment = deploy_mongodb(name="jsonbench-travel", port=27127)
couchbase_deployment = deploy_couchbase(
    name="jsonbench-travel-cb",
    management_port=18291,
    query_port=18293,
    kv_port=21230,
    bucket="travel",
)
print(mongo_deployment.uri)
print(couchbase_deployment.console_url)

## Load the same dataset into both databases

Load each collection into MongoDB and Couchbase, preserving document IDs and date strings.
BONDI derives Couchbase keys from the IDs and creates no query indexes.
The default `create` mode requires empty collections. Run cleanup before repeating deployment and loading.

In [ ]:
from bondi.couchbase import CouchbaseCollectionRef
from bondi.data import load_documents
from bondi.mongodb import MongoCollectionRef

for name, documents in dataset.items():
    mongo_result = load_documents(
        documents,
        target=mongo_deployment,
        collection=MongoCollectionRef("travel", name),
    )
    couchbase_result = load_documents(
        documents,
        target=couchbase_deployment,
        collection=CouchbaseCollectionRef("travel", "_default", name),
    )
    print(mongo_result.destinations, couchbase_result.destinations)

## Build Q1–Q4 workloads

Translate all four templates into one ordered BONDI workload per database.

In [ ]:
from pprint import pprint

from bondi.workload import Workload, WorkloadEntry

from json_bench.queries import translate_couchbase, translate_mongodb

q1 = {
    "name": "Query 1",
    "description": "Total booking value per tour",
    "primary_collection": "bookings",
    "operations": {"group": {"by": "tour_id", "aggregates": {"total_revenue": "sum(total_price)"}}},
}

q2 = {
    "name": "Query 2",
    "description": "Traveller age distribution for each tour",
    "primary_collection": "bookings",
    "operations": {
        "unnest": [{"field": "travellers", "from": "bookings"}],
        "group": {
            "by": "tour_id",
            "aggregates": {
                "avg_age": "avg(travellers.age)",
                "min_age": "min(travellers.age)",
                "max_age": "max(travellers.age)",
            },
        },
    },
}

q3 = {
    "name": "Query 3",
    "description": "Average traveller group size (per booking) for each tour",
    "primary_collection": "bookings",
    "operations": {
        "group": {"by": "tour_id", "aggregates": {"avg_grp_size": "avg(size(travellers))"}}
    },
}

q4 = {
    "name": "Query 4",
    "description": "Number of itinerary visits to a specified location, grouped by tour difficulty",
    "primary_collection": "tours",
    "operations": {
        "unnest": ["itinerary"],
        "match": {"itinerary.location": "Donnafurt"},
        "group": {"by": "difficulty", "aggregates": {"num_of_tours": "count(*)"}},
    },
}

# Choose a location that occurs in this small generated dataset.
q4["operations"]["match"]["itinerary.location"] = dataset["tours"][0]["itinerary"][0]["location"]
print("Q4 location:", q4["operations"]["match"]["itinerary.location"])

templates = [q1, q2, q3, q4]


mongo_workload = Workload(
    [WorkloadEntry(translate_mongodb(t, database="travel")) for t in templates]
)
couchbase_workload = Workload(
    [WorkloadEntry(translate_couchbase(t, bucket="travel")) for t in templates]
)

for template, mongo_entry, couchbase_entry in zip(
    templates, mongo_workload, couchbase_workload, strict=True
):
    print(template["name"])
    pprint(mongo_entry.query.pipeline)
    print(couchbase_entry.query.statement)

## Execute the workloads

Run each query once and retain its documents. Create the original bookings and tours indexes first.
Use REQUEST_PLUS to include newly loaded data in Couchbase. This is a correctness check, not a benchmark.

In [ ]:
from bondi.couchbase import CouchbaseExecutor, CouchbaseQuery
from bondi.mongodb import MongoExecutor
from couchbase.n1ql import QueryScanConsistency

with MongoExecutor.from_uri(mongo_deployment.uri, timeout_seconds=60) as executor:
    mongo_results = executor.run_workload(mongo_workload, collect_results=True)

with CouchbaseExecutor.from_uri(
    couchbase_deployment.uri,
    username=couchbase_deployment.username,
    password=couchbase_deployment.password,
    timeout_seconds=60,
) as executor:
    for statement in [
        "CREATE INDEX `bookings-tour_id` IF NOT EXISTS ON `bookings`(`tour_id`);",
        "CREATE INDEX `tours-id` IF NOT EXISTS ON `tours`(`_id`);",
    ]:
        executor.run(
            CouchbaseQuery(statement, query_context=couchbase_workload[0].query.query_context)
        )
    couchbase_results = executor.run_workload(
        couchbase_workload,
        collect_results=True,
        query_kwargs={"scan_consistency": QueryScanConsistency.REQUEST_PLUS},
    )

## Compare query results

Compare each query in workload order. Require the same tour groups and aggregate fields,
ignore row order, and allow small floating-point differences.

In [ ]:
from math import isclose

for template, mongo_result, couchbase_result in zip(
    templates, mongo_results, couchbase_results, strict=True
):
    assert mongo_result is not None and couchbase_result is not None, "Query did not run"
    group_field = template["operations"]["group"]["by"]
    mongo_rows = {row["_id"]: row for row in mongo_result.documents}
    couchbase_rows = {row[group_field]: row for row in couchbase_result.documents}
    assert len(mongo_rows) == len(mongo_result.documents), "Duplicate MongoDB groups"
    assert len(couchbase_rows) == len(couchbase_result.documents), "Duplicate Couchbase groups"
    assert mongo_rows, "Query returned no tour groups"
    assert mongo_rows.keys() == couchbase_rows.keys(), "Group keys differ"

    aggregates = template["operations"]["group"]["aggregates"]
    for group_key, mongo_row in mongo_rows.items():
        couchbase_row = couchbase_rows[group_key]
        assert mongo_row.keys() == {"_id", *aggregates}, "MongoDB fields differ"
        assert couchbase_row.keys() == {group_field, *aggregates}, "Couchbase fields differ"
        for field in aggregates:
            assert isclose(mongo_row[field], couchbase_row[field], rel_tol=1e-9, abs_tol=1e-6), (
                f"{template['name']}, {group_key}, {field}: "
                f"{mongo_row[field]} vs {couchbase_row[field]}"
            )
    print(f"{template['name']}: results match for all {len(mongo_rows)} groups.")

## Clean up when finished

Remove both deployments, including their data and saved configuration.

In [ ]:
mongo_deployment.remove()
couchbase_deployment.remove()